In [1]:
# Run in Colab Notebook
!pip install scikit-learn pandas numpy joblib spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np

# Load Open Dataset (Or create a initial training array based on PROMISE dataset structure)
data = {
    "user_story": [
        "As a user, I want to process credit card payments securely so that I can purchase items.",
        "As an admin, I want to manage system permissions and revoke user access.",
        "As a user, I want to update my profile picture and personal info.",
        "As a user, I want to search and filter products by category and price range.",
        "As a user, I want to view footer links aligned in the center of the screen.",
        "As a visitor, I want to see a copyright notice at the bottom of the landing page.",
        "As a user, I want to reset my lost password via SMS token verification.",
        "As a customer, I want to receive email notifications when my order status changes."
    ],
    "priority": ["High", "High", "Medium", "Medium", "Low", "Low", "High", "Medium"]
}

df = pd.DataFrame(data)
print("Dataset Sample:")
print(df.head())

Dataset Sample:
                                          user_story priority
0  As a user, I want to process credit card payme...     High
1  As an admin, I want to manage system permissio...     High
2  As a user, I want to update my profile picture...   Medium
3  As a user, I want to search and filter product...   Medium
4  As a user, I want to view footer links aligned...      Low


In [3]:
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    # Lowercase & remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    # Lemmatize text using spaCy
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(lemmas)

df['clean_story'] = df['user_story'].apply(preprocess_text)

# Feature Extraction
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
X = vectorizer.fit_transform(df['clean_story']).toarray()
y = df['priority']

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Classifier
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Evaluate Baseline
y_pred = model.predict(X_test)
print(f"Model Training Accuracy: {accuracy_score(y_test, y_pred) * 100}%")

Model Training Accuracy: 0.0%


In [6]:
def generate_scenarios(user_story):
    """
    Parses Agile User Story structure and outputs 3 scenario types:
    Positive, Negative, Edge Case.
    """
    # Regex structure extraction: "As a [role], I want to [action] so that [benefit]"
    match = re.search(r"As a (.*?),\s*I want to (.*?)(?:\s*so that (.*))?$", user_story, re.IGNORECASE)

    if match:
        role = match.group(1).strip()
        action = match.group(2).strip()
    else:
        role = "User"
        action = user_story

    scenarios = [
        # Positive
        {
            "Type": "Positive",
            "Scenario": f"Verify successful execution of {action}",
            "Steps": f"1. Log in as {role}\n2. Perform action: {action}\n3. Confirm valid response.",
            "Expected Result": "Action completed successfully with 200 OK state."
        },
        # Negative
        {
            "Type": "Negative",
            "Scenario": f"Attempt {action} with missing mandatory inputs",
            "Steps": f"1. Navigate to {action} form\n2. Submit blank mandatory fields.",
            "Expected Result": "System prevents submission and shows field validation error."
        },
        # Edge Case
        {
            "Type": "Edge Case",
            "Scenario": f"Execute {action} under network disconnect / timeout limit",
            "Steps": f"1. Trigger {action}\n2. Interrupt network connection mid-request.",
            "Expected Result": "System handles network error gracefully without duplicating records."
        }
    ]
    return scenarios

In [8]:
import joblib

# Export model and vectorizer weights to binary files
joblib.dump(model, 'priority_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Model artifacts saved successfully!")

Model artifacts saved successfully!


In [9]:
import joblib
from google.colab import files

# 1. Save the model files inside Colab
joblib.dump(model, 'priority_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Model artifacts saved successfully!")

# 2. Automatically download files to your local computer
files.download('priority_model.pkl')
files.download('tfidf_vectorizer.pkl')

Model artifacts saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# Balanced Open Dataset Sample
data = {
    "user_story": [
        # HIGH PRIORITY (Security, Auth, Financial, Data Loss)
        "As a user, I want to process credit card payments securely.",
        "As an admin, I want to revoke user permissions and access logs.",
        "As a customer, I want to reset my lost account password via SMS.",
        "User should be able to login to home using login interface",
        "System must encrypt sensitive personal health and billing details.",

        # MEDIUM PRIORITY (Core Features, Search, Filters, Profile)
        "As a user, I want to filter products by category and price.",
        "As a user, I want to update my profile picture and bio.",
        "As a customer, I want to receive email notifications on order status.",
        "User can export search results to a PDF file.",
        "User can add items to their shopping cart list.",

        # LOW PRIORITY (UI/UX, Formatting, Alignment, Styling)
        "Align footer copyright text in the center of the page.",
        "Change submit button color to blue on hover state.",
        "Display app logo at the top left corner of navbar.",
        "As a user, I want to see tooltips when hovering over icons.",
        "Update font style of the user agreement terms page."
    ],
    "priority": [
        "High", "High", "High", "High", "High",
        "Medium", "Medium", "Medium", "Medium", "Medium",
        "Low", "Low", "Low", "Low", "Low"
    ]
}

df = pd.DataFrame(data)

# Vectorize & Train
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform(df['user_story']).toarray()
y = df['priority']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Save and download new models
joblib.dump(model, 'priority_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Retrained successfully! Download and upload the new .pkl files to GitHub.")

Retrained successfully! Download and upload the new .pkl files to GitHub.


In [11]:
from google.colab import files

files.download('priority_model.pkl')
files.download('tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>